In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


TRAIN_MIN_YEAR = 1964
TRAIN_MAX_YEAR = 1993
TEST_MIN_YEAR = 1994
TEST_MAX_YEAR = 2016

BASE_DIR = os.getcwd()
DATA_DIR = "data"

feature_cols = ["LME", "BEME", "r12_2", "OP", "Investment","ST_Rev", "LT_Rev", "AC", "IdioVol", "LTurnover"]

N_ESTIMATORS = 200
MAX_DEPTH = 4
MIN_SAMPLES_LEAF = 20
MAX_FEATURES = "sqrt" # sqrt, log2 or None
RANDOM_STATE = 23


OUT_DIR2 = os.path.join(
    BASE_DIR,
    f"results_ne{N_ESTIMATORS}_md{MAX_DEPTH}_mf{MAX_FEATURES}"
)
os.makedirs(OUT_DIR2, exist_ok=True)

def load_all(data_dir: str):
    csv_files = sorted(glob.glob(os.path.join(data_dir, "y*.csv")))
    dfs = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        if not re.match(r"y\d{4}\.csv", fname):
            continue
        dfs.append(pd.read_csv(fpath))
    if not dfs:
        raise ValueError("No y*.csv files found.")
    return pd.concat(dfs, ignore_index=True)

df_all = load_all(DATA_DIR)
train_mask = (df_all["yy"] >= TRAIN_MIN_YEAR) & (df_all["yy"] <= TRAIN_MAX_YEAR)
df_train = df_all.loc[train_mask].copy()
if df_train.empty:
    raise ValueError("No rows found in the selected training interval.")

X_train = df_train
y_train = df_train["ret"]

test_mask = (df_all["yy"] >= TEST_MIN_YEAR) & (df_all["yy"] <= TEST_MAX_YEAR)
df_test = df_all.loc[test_mask].copy().reset_index(drop=True)
if df_test.empty:
    raise ValueError("No rows found in the selected test interval.")  

X_test = df_test
y_test = df_test["ret"]

In [ ]:
from median_split_forest import MedianSplitForestExporter
import joblib

exporter = MedianSplitForestExporter(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        max_features=MAX_FEATURES,
        bootstrap=True,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

exporter.fit_and_export(
        X=X_train,
        y=y_train,
        feature_cols=feature_cols,
        out_nodes_csv=os.path.join(OUT_DIR2, f"nodes.csv")
    )

exporter.fit_test(
        X_test=X_test,
        feature_cols=feature_cols,      # same order as in training
    )

# Or explicitly delete per Tree/Node
for tree in exporter.trees_:
    for node in tree.nodes_:
        node.idx = None
        node.monthly_excess_returns = None

model_path = os.path.join(OUT_DIR2, "fitted_exporter.joblib")
joblib.dump(exporter, model_path)

In [ ]:
from SR import compute_excess
import joblib
import pickle


K = 100

model_path = os.path.join(OUT_DIR2, "fitted_exporter.joblib")
exporter = joblib.load(model_path)

no_portfolios = 1000

# Reproducible random selection
rng = np.random.default_rng(42)

save_path = os.path.join(OUT_DIR2, "top_items_by_row_by_k.pkl")

if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        top_items_by_row_by_k = pickle.load(f)
else:
    top_items_by_row_by_k = {}

# Generate month codes based on X_test
if "yy" not in X_test.columns or "mm" not in X_test.columns:
    raise KeyError("X_test must contain the 'yy' and 'mm' columns.")

unique_months = (
    X_test[["yy", "mm"]]
    .drop_duplicates()
    .sort_values(["yy", "mm"])
    .reset_index(drop=True)
)
month_index = {(int(y), int(m)): i for i, (y, m) in unique_months.iterrows()}
month_codes = np.array([
    month_index[(int(y), int(m))] for y, m in zip(X_test["yy"], X_test["mm"])
])

first_code = 0  # codes follow the order of unique_months
month_rows_first = np.where(month_codes == first_code)[0]
if month_rows_first.size == 0:
    raise ValueError("First month has no rows; cannot sample subset.")

sample_size = min(no_portfolios, month_rows_first.size)
subset_idx = rng.choice(month_rows_first, size=sample_size, replace=False).tolist()
print(f"Randomly sampled {len(subset_idx)} rows from the first month.")
if not subset_idx:
    raise ValueError("No rows selected from first month subset.")
    
# 1) many calls, reuse the same accumulator
mean_output_path = os.path.join(OUT_DIR2, f"ret_k{K}.csv")
acc = None
top_items_by_row = {}

for row_id in subset_idx:
    top_items, acc = exporter.top_k_neighbors(
        k = K,
        X=X_test,
        feature_cols=feature_cols,
        y=y_test,
        row_id=row_id,
        mean_output_df=acc,               # Accumulator
        return_mean_output_df=True        # return updated accumulator
    )
    top_items_by_row[row_id] = top_items

top_items_by_row_by_k[K] = top_items_by_row

with open(save_path, "wb") as f:
    pickle.dump(top_items_by_row_by_k, f)

# 2) write once
acc.to_csv(mean_output_path, index=True)
compute_excess(
    nodes_path=mean_output_path,
    out_path=os.path.join(OUT_DIR2, f"EXCESS_ret_k{K}.csv")
)